In [7]:
import spacy 


import os
import sys
import dotenv

import json


from collections import defaultdict
from prettytable import PrettyTable

dotenv.load_dotenv()
ROOT_DIR = os.environ.get("ROOT_DIR")
sys.path.append(f"{ROOT_DIR}/scripts")

from evaluation import evaluate

In [8]:
test_before_2000 = json.load(open(f"{ROOT_DIR}/data/splits/test_before_2000.json", "r"))
test_after_2000 = json.load(open(f"{ROOT_DIR}/data/splits/test_after_2000.json", "r"))

# Spacy without training

# Inference

In [9]:
nlp = spacy.load("fr_core_news_sm")

In [10]:
def inference(nlp, data):
    all_doc = []
    for doc in data:
        entities = []
        text = doc["texte"]
        doc["predicted_entities"] = []
        spacy_doc = nlp(text)
        for ent in spacy_doc.ents:
            entities.append({
                "texte": ent.text,
                "tag": ent.label_,
                "debut": ent.start_char,
                "fin": ent.end_char})

        all_doc.append({
            "id": doc["id"],
            "annee": doc["annee"],
            "predicted_entities": entities
        })

    return all_doc

In [11]:
inference_before_2000 = inference(nlp, test_before_2000)
inference_after_2000 = inference(nlp, test_after_2000)   

# Evaluation

In [12]:
metrics_before_2000 = evaluate(inference_before_2000, test_before_2000)

Global NER Performance (Exact Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.0775 |
|   Recall  | 0.2448 |
|  F1-Score | 0.1177 |
+-----------+--------+

Global NER Performance (Partial Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.1572 |
|   Recall  | 0.4965 |
|  F1-Score | 0.2388 |
+-----------+--------+

Performance by Tag — Exact Match (MISC excluded)
+------+-----------+--------+----------+---------+------------+
| Tag  | Precision | Recall | F1-Score | Support | Partial TP |
+------+-----------+--------+----------+---------+------------+
| LOC  |   0.0571  | 0.2346 |  0.0918  |    81   |     34     |
| MISC |   0.0096  | 0.0488 |  0.0161  |    82   |     8      |
| ORG  |   0.1019  | 0.2132 |  0.1379  |   197   |     42     |
| PER  |   0.2062  | 0.5797 |  0.3042  |    69   |     24     |
+------+-----------+--------+----------+---------+------------+


In [13]:
metrics_after_2000 = evaluate(inference_after_2000, test_after_2000)

Global NER Performance (Exact Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.0260 |
|   Recall  | 0.4545 |
|  F1-Score | 0.0491 |
+-----------+--------+

Global NER Performance (Partial Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.0349 |
|   Recall  | 0.6116 |
|  F1-Score | 0.0661 |
+-----------+--------+

Performance by Tag — Exact Match (MISC excluded)
+------+-----------+--------+----------+---------+------------+
| Tag  | Precision | Recall | F1-Score | Support | Partial TP |
+------+-----------+--------+----------+---------+------------+
| LOC  |   0.0398  | 0.5000 |  0.0738  |    42   |     6      |
| MISC |   0.0000  | 0.0000 |  0.0000  |    10   |     1      |
| ORG  |   0.0401  | 0.7692 |  0.0762  |    26   |     5      |
| PER  |   0.0308  | 0.3256 |  0.0563  |    43   |     7      |
+------+-----------+--------+----------+---------+------------+


# Trained Spacy

In [17]:
training_data_raw = json.load(open(f"{ROOT_DIR}/data/splits/train.json", "r"))

In [22]:
training_data = {'classes' : ['PER', 'ORG', 'MISC', 'LOC'], 'annotations' : []}
for doc in training_data_raw:
  temp_dict = {}
  temp_dict['text'] = doc['texte']
  temp_dict['entites'] = []
  for annotation in doc['entites']:
    debut = annotation['debut']
    fin = annotation['fin']
    tag = annotation['tag'].upper()
    temp_dict['entites'].append((debut, fin, tag))
  training_data['annotations'].append(temp_dict)
  
print(training_data['annotations'][0]['entites'])

[(0, 11, 'ORG'), (14, 21, 'ORG'), (82, 95, 'PER'), (113, 116, 'LOC'), (117, 136, 'LOC'), (138, 153, 'LOC'), (187, 223, 'ORG'), (246, 279, 'ORG'), (394, 405, 'LOC'), (535, 553, 'LOC'), (1001, 1019, 'PER'), (1227, 1233, 'LOC'), (1939, 1945, 'LOC'), (2086, 2092, 'LOC'), (2180, 2191, 'LOC'), (2292, 2298, 'LOC'), (2419, 2425, 'LOC'), (2499, 2505, 'LOC'), (2522, 2528, 'LOC'), (2734, 2759, 'MISC'), (2760, 2778, 'MISC'), (2779, 2793, 'MISC'), (2834, 2845, 'MISC'), (2846, 2859, 'MISC'), (2854, 2858, 'LOC'), (2860, 2918, 'MISC'), (2972, 2985, 'LOC'), (1191, 1202, 'ORG'), (1215, 1222, 'ORG')]


## Training

## Evaluation